## 1) CPJudgeBench Overview

CPJudgeBench is designed as a benchmark resource for evaluating LLM-as-a-judge in constraint programming.

It is organized around parameterized CP problem families. Each benchmark item includes:
- a problem description,
- a ground-truth reference CP model,
- generated instances,
- candidate CP models,
- solver-supported oracle information, and
- judge prompts.

Candidate models are generated across different CP languages or frameworks (such as MiniZinc, PyCSP3, and CPMpy), using different generator LLMs and target correctness labels.

Instead of evaluating candidates only by SAT/UNSAT agreement, single-solution validity, or objective-value matching, the benchmark labels each candidate according to its relationship with the intended solution space:
- exact equivalence,
- unsoundness,
- incompleteness,
- combined unsoundness and incompleteness,
- status-only correctness, or
- non-executability.

This enables evaluation of whether LLM judges can reason about declarative CP models, detect semantic modeling errors, and compare candidates at the level of solution-space correctness.

In [2]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import contextlib
import io
import json
import os
import re
import subprocess
import tempfile

import dotenv
import numpy as np
import pandas as pd
from cpmpy import Model
from langchain.chat_models import init_chat_model

In [26]:
correctness_labels = [
    "equivalent",
    "unsound",
    "incomplete",
    "unsound-incomplete",
    "non-executable",
    "status-only correct",
]

language_labels = ["minizinc", "CPMpy",
        # "pyCSP3",
]

# OpenRouter model ids (provider/model[:tag])
models_set = [
    "openai/gpt-5.1-codex-mini",
    "openai/gpt-4o-mini",
    # "anthropic/claude-sonnet-4.5",
    # "openai/gpt-5.1:thinking",
]


n_instances = 5
solution_limit = 100000
time_limit_cpmpy_sec = 120
time_limit_minizinc_sec = 120

In [4]:
# def load_jsonl(path: Path) -> list[dict[str, Any]]:
#     rows: list[dict[str, Any]] = []
#     with path.open("r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if line:
#                 rows.append(json.loads(line))
#     return rows


# def get_problem_by_id(records: list[dict[str, Any]], problem_id: str) -> dict[str, Any]:
#     df = pd.DataFrame(records)
#     matches = np.where(df["id"] == problem_id)[0]
#     if len(matches) == 0:
#         raise ValueError(f"Problem id not found: {problem_id}")
#     return records[int(matches[0])]


# jsonl_path = Path("dcp-bench-open.jsonl")
# records = load_jsonl(jsonl_path)
# print(f"Loaded {len(records)} benchmark records")


In [5]:

# targeted_id = "session2_subsets_100"
# sample = get_problem_by_id(records, targeted_id)

# problem_description = sample.get("description", "")
# reference_cp_model = sample.get("model", "")
# example_instance = sample.get("example_instance", "")
# data_instances = sample.get("instances", [])
# decision_variables = sample.get("decision_variables", [])
# example_solution = sample.get("example_solution", {})

# print(f"Problem id: {sample.get('id')}")
# print(f"Decision variables: {decision_variables}")
# print(f"Has explicit instances: {bool(data_instances)}")
# print(f"Example solution keys: {list(example_solution.keys()) if isinstance(example_solution, dict) else 'n/a'}")

In [6]:
targeted_id = "domino_tiling"

problem_description = """
Consider an m x n rectangular chessboard. We want to tile this board with dominoes,
where each domino is a 2 x 1 rectangle. A tiling is a placement of dominoes such that
every square of the board is covered exactly once, no dominoes overlap, and no domino
extends beyond the boundary of the board.

Print one valid tiling of the chessboard.
"""

reference_cp_model = """
from cpmpy import *

h = boolvar(shape=(m, n - 1), name="h")  # horizontal domino starts
v = boolvar(shape=(m - 1, n), name="v")  # vertical domino starts

model = Model()
for i in range(m):
    for j in range(n):
        cover = []
        if j > 0:
            cover.append(h[i, j - 1])
        if j < n - 1:
            cover.append(h[i, j])
        if i > 0:
            cover.append(v[i - 1, j])
        if i < m - 1:
            cover.append(v[i, j])
        model += (sum(cover) == 1)

if model.solve():
    print(h.value())
    print(v.value())
"""

example_instance = """
m = 4
n = 6
"""

data_instances = []

decision_variables = ["h", "v"]

In [27]:
@dataclass
class ModelSpec:
    provider: str
    name: str
    tag: str | None = None
    raw: str = ""

    @classmethod
    def from_string(cls, model_str: str) -> "ModelSpec":
        if "/" not in model_str:
            raise ValueError("Model must be '<provider>/<model>[:tag]'")
        provider, rest = model_str.split("/", 1)
        if ":" in rest:
            name, tag = rest.split(":", 1)
        else:
            name, tag = rest, None
        return cls(provider=provider, name=name, tag=tag, raw=model_str)

    def openrouter_model_name(self) -> str:
        return f"{self.name}:{self.tag}" if self.tag else self.name


model_specs = [ModelSpec.from_string(m) for m in models_set]
model_specs

[ModelSpec(provider='openai', name='gpt-5.1-codex-mini', tag=None, raw='openai/gpt-5.1-codex-mini'),
 ModelSpec(provider='openai', name='gpt-4o-mini', tag=None, raw='openai/gpt-4o-mini')]

In [8]:
dotenv.load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError("OPENROUTER_API_KEY is not set in environment/.env")


def get_openrouter_llm(spec: ModelSpec):
    return init_chat_model(
        model=spec.openrouter_model_name(),
        model_provider=spec.provider,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )


output_dir = Path("data-storage") / f"candidates-{targeted_id}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {output_dir}")

Output dir: data-storage\candidates-domino_tiling


In [9]:
def extract_json_array(text: str):
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
    except Exception:
        pass

    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON array found in model output")
    return json.loads(text[start : end + 1])


def llm_response_to_text(resp: Any) -> str:
    """Normalize LangChain/OpenAI response content to plain text.
    Some models (e.g., GPT-5 family) may return content as a list of blocks.
    """
    content = getattr(resp, "content", resp)

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                txt = block.get("text")
                if isinstance(txt, str) and txt:
                    parts.append(txt)
                elif isinstance(block.get("content"), str):
                    parts.append(block["content"])
            elif hasattr(block, "text") and isinstance(getattr(block, "text"), str):
                parts.append(getattr(block, "text"))

        if parts:
            return "\n".join(parts)
        return str(content)

    if isinstance(content, dict):
        if isinstance(content.get("text"), str):
            return content["text"]
        if isinstance(content.get("content"), str):
            return content["content"]

    return str(content)


def extract_code_block(text: str) -> str:
    if "```" not in text:
        return text.strip()
    parts = text.split("```")
    candidate = max(parts[1::2], key=len).strip() if len(parts) >= 3 else text.strip()
    lines = candidate.splitlines()
    if lines and lines[0].strip().lower() in {"python", "minizinc"}:
        lines = lines[1:]
    return "\n".join(lines).strip()

In [ ]:
def generate_instances(spec: ModelSpec, count: int = 5) -> list[dict[str, Any]]:
    llm = get_openrouter_llm(spec)
    prompt = f"""
You are an expert in constraint programming benchmark construction.
Your task is to create high-quality test instances for one CP problem.

Problem description:
{problem_description}

Reference model:
{reference_cp_model}

Reasoning requirements (think before writing output):
1) Infer the true input schema (field names, types, shape constraints, value ranges).
2) Infer hidden invariants implied by the model (e.g., dimensions align, capacities nonnegative, domains meaningful).
3) Construct a compact but comprehensive set of instances covering:
   - typical valid case,
   - boundary/minimal sizes,
   - near-degenerate but valid structures,
   - stress/large-value case still solver-feasible,
   - corner case likely to expose modeling mistakes.
4) Keep instances realistic and internally consistent.

Output requirements (strict):
- Return EXACTLY {count} instances.
- Return ONLY a JSON array of objects.
- No markdown, no prose, no code fences.
- Every object must include all required input fields.
- Add '_case_note' (short string) describing what that instance is testing.
""".strip()

    text = llm_response_to_text(llm.invoke(prompt))
    instances = extract_json_array(text)
    if not (3 <= len(instances) <= 5):
        raise ValueError(f"Expected 3-5 instances, got {len(instances)}")
    return instances


instance_generation_results: dict[str, dict[str, Any]] = {}
for spec in model_specs:
    model_label = spec.raw
    out_path = Path(f"data-storage/instances-{targeted_id}-{model_label.replace('/', '_')}.json")
    if out_path.exists():
        print(f"Skip existing: {out_path}")
        continue

    try:
        generated = generate_instances(spec, count=n_instances)
        out_path.write_text(json.dumps(generated, ensure_ascii=False, indent=2), encoding="utf-8")
        instance_generation_results[model_label] = {"ok": True, "count": len(generated), "path": str(out_path)}
    except Exception as e:
        instance_generation_results[model_label] = {"ok": False, "error": str(e)}
    
    break

instance_generation_results



Skip existing: data-storage\instances-domino_tiling-openai_gpt-5.1-codex-mini.json


{}

In [11]:
label_guidance = {
    "equivalent": "Semantically equivalent to the intended model.",
    "unsound": "Allows invalid solutions (false positives).",
    "incomplete": "Misses valid solutions (false negatives).",
    "unsound-incomplete": "Has both false positives and false negatives.",
    "non-executable": "Syntactically/API invalid and cannot execute.",
    "status-only correct": "Likely preserves SAT/UNSAT status but not exact solution space.",
}


def generate_candidate_model(spec: ModelSpec, language: str, label: str, feedback_notes: str = "") -> str:
    llm = get_openrouter_llm(spec)
    prompt = f"""
You are generating ONE candidate CP model for benchmarking model-judging quality.

Problem description:
{problem_description}

Reference model (for understanding only):
{reference_cp_model}

Example data instance:
{example_instance}

Target language: {language}
Target correctness label: {label}
Target behavior: {label_guidance[label]}

Design instructions:
1) First infer intended decision variables, constraints, and objective/status semantics.
2) Before writing code, do an internal reasoning pass about this specific problem and identify concrete ways to shape the solution space to match the target label.
3) Derive issue mechanisms from the provided problem/reference model only (do NOT rely on canned patterns or memorized example mistakes).
4) Use that internal reasoning plan to write a model in the target language that intentionally matches the requested label behavior.
5) Keep variable names aligned with the problem statement (especially decision variables used in evaluation).
6) Ensure code is self-contained for one instance (use given instance fields directly).

Reasoning protocol (must follow before generation):
- Think through boundary/index coverage, mutual-exclusion logic, and completeness of constraint families for this problem instance format.
- For labels with semantic mismatch (unsound/incomplete/unsound-incomplete/status-only correct), explicitly decide which valid solutions are removed and/or which invalid solutions are admitted.
- Ensure the planned mismatch is deliberate and non-trivial, while keeping the model executable unless label is non-executable.
- Keep this reasoning internal; do not output explanation.

Label-specific precision:
- equivalent: same solution space as intended model.
- unsound: admits at least one invalid solution but should still solve.
- incomplete: excludes at least one valid solution but should still solve.
- unsound-incomplete: both admits invalid and excludes valid solutions.
- status-only correct: tends to preserve SAT/UNSAT status while not preserving exact solution space.
- non-executable: intentionally non-runnable (syntax/API/semantic construction error).

Feedback from previous attempt (if any):
{feedback_notes if feedback_notes else 'None'}

If feedback is provided, correct the previous mistakes while preserving the requested target label behavior.

Output requirements (strict):
- Return ONLY raw code.
- No explanation, no markdown fences.
- If language is CPMpy or pyCSP3, return Python code.
- If language is MiniZinc, return valid .mzn text.
""".strip()
    text = llm_response_to_text(llm.invoke(prompt))
    return extract_code_block(text)

# Candidate creation + validation + feedback loop is executed in a later combined cell.
# This cell intentionally keeps only prompt/function definitions.

In [12]:
# Reference model preview (optional)
print(reference_cp_model[:1200])


from cpmpy import *

h = boolvar(shape=(m, n - 1), name="h")  # horizontal domino starts
v = boolvar(shape=(m - 1, n), name="v")  # vertical domino starts

model = Model()
for i in range(m):
    for j in range(n):
        cover = []
        if j > 0:
            cover.append(h[i, j - 1])
        if j < n - 1:
            cover.append(h[i, j])
        if i > 0:
            cover.append(v[i - 1, j])
        if i < m - 1:
            cover.append(v[i, j])
        model += (sum(cover) == 1)

if model.solve():
    print(h.value())
    print(v.value())



In [13]:
# Validation helpers (problem-agnostic)

def instance_to_dict(example_instance_text: str, data_instances_list: list[Any]) -> dict[str, Any]:
    if isinstance(data_instances_list, list) and data_instances_list and isinstance(data_instances_list[0], dict):
        return data_instances_list[0]

    txt = str(example_instance_text).strip()
    if not txt:
        return {}

    local_ns: dict[str, Any] = {}
    exec(txt, {}, local_ns)
    return {k: v for k, v in local_ns.items() if not k.startswith("__")}


def flatten_cp_vars(v):
    if isinstance(v, (list, tuple)):
        out = []
        for x in v:
            out.extend(flatten_cp_vars(x))
        return out
    if isinstance(v, np.ndarray):
        return [x for x in v.flat]
    if hasattr(v, "flatten") and not hasattr(v, "value"):
        try:
            return list(v.flatten())
        except Exception:
            pass
    return [v]


def extract_assignment_key(var_names: list[str], namespace: dict[str, Any]):
    key_parts = []
    for name in var_names:
        vars_flat = flatten_cp_vars(namespace[name])
        vals = tuple(int(v.value()) for v in vars_flat)
        key_parts.append((name, vals))
    return tuple(key_parts)


def observed_label_from_sets(ref_space: set, cand_space: set):
    fp = len(cand_space - ref_space)
    fn = len(ref_space - cand_space)
    if fp == 0 and fn == 0:
        observed = "equivalent"
    elif fp > 0 and fn == 0:
        observed = "unsound"
    elif fp == 0 and fn > 0:
        observed = "incomplete"
    else:
        observed = "unsound-incomplete"
    return observed, fp, fn


def label_match_rule(claimed: str, observed: str, ref_space: set, cand_space: set) -> bool:
    if claimed == "status-only correct":
        return (len(ref_space) > 0) == (len(cand_space) > 0) and observed != "equivalent"
    if claimed == "non-executable":
        return False
    return claimed == observed


In [14]:
def execute_cpmpy_and_enumerate(
    code_text: str,
    instance_dict: dict[str, Any],
    decision_var_names: list[str],
    solution_limit: int = solution_limit,
    time_limit: int = time_limit_cpmpy_sec,
):
    """Execute CPMpy code and enumerate complete solution-space via solveAll(display=...)."""
    namespace: dict[str, Any] = {}
    namespace.update(instance_dict)

    try:
        with contextlib.redirect_stdout(io.StringIO()):
            exec(code_text, namespace, namespace)

        if "model" not in namespace:
            return {"ok": False, "error": "No `model` object found", "space": None, "num_solutions": 0}

        model = namespace["model"]
        if not isinstance(model, Model):
            return {"ok": False, "error": "`model` is not CPMpy Model", "space": None, "num_solutions": 0}

        for dv in decision_var_names:
            if dv not in namespace:
                return {"ok": False, "error": f"Decision variable `{dv}` missing", "space": None, "num_solutions": 0}

        space: set = set()
        num_solutions = [0]

        def _display_solution():
            num_solutions[0] += 1
            space.add(extract_assignment_key(decision_var_names, namespace))

        model.solveAll(display=_display_solution, solution_limit=solution_limit, time_limit=time_limit)

        if num_solutions[0] >= solution_limit:
            return {
                "ok": True,
                "error": f"Reached solution_limit={solution_limit}; full space may be truncated",
                "space": space,
                "num_solutions": num_solutions[0],
            }

        return {"ok": True, "error": "", "space": space, "num_solutions": num_solutions[0]}

    except SyntaxError as e:
        src_line = (e.text or "").strip()
        detail = f"SyntaxError at line {e.lineno}, offset {e.offset}: {e.msg}"
        if src_line:
            detail += f" | source: {src_line}"
        return {"ok": False, "error": detail, "space": None, "num_solutions": 0}
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}", "space": None, "num_solutions": 0}


In [15]:
def _check_minizinc_syntax(model_path: Path) -> tuple[bool, str]:
    try:
        result = subprocess.run(
            ["minizinc", "--compile", "--solver", "gecode", str(model_path)],
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        msg = "MiniZinc executable not found; syntax not checked."
        return True, msg

    ok = result.returncode == 0
    error_output = (result.stderr or "") + (result.stdout if not ok else "")
    return ok, error_output.strip()


def _flatten_value_to_int_tuple(v):
    if isinstance(v, (list, tuple)):
        out = []
        for x in v:
            out.extend(_flatten_value_to_int_tuple(x))
        return tuple(out)
    if isinstance(v, bool):
        return (1 if v else 0,)
    if isinstance(v, (int, np.integer)):
        return (int(v),)
    try:
        return (int(v),)
    except Exception:
        return (hash(str(v)),)


def _extract_json_objects(text: str):
    objs = []
    dec = json.JSONDecoder()
    i = 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        try:
            obj, j = dec.raw_decode(text, i)
            objs.append(obj)
            i = j
        except Exception:
            i += 1
    return objs


def _extract_solutions_from_mzn_json_stream(stdout_text: str, decision_var_names: list[str]):
    space = set()
    for obj in _extract_json_objects(stdout_text):
        candidate = None
        if isinstance(obj, dict):
            if isinstance(obj.get("output"), dict) and isinstance(obj["output"].get("json"), dict):
                candidate = obj["output"]["json"]
            elif isinstance(obj.get("solution"), dict):
                candidate = obj["solution"]
            elif all(k in obj for k in decision_var_names):
                candidate = obj

        if isinstance(candidate, dict) and all(k in candidate for k in decision_var_names):
            key = tuple((name, _flatten_value_to_int_tuple(candidate[name])) for name in decision_var_names)
            space.add(key)
    return space


def execute_minizinc_and_enumerate(
    code_text: str,
    decision_var_names: list[str],
    solution_limit: int = solution_limit,
    time_limit_sec: int = time_limit_minizinc_sec,
):
    with tempfile.TemporaryDirectory() as td:
        model_path = Path(td) / "candidate.mzn"
        model_path.write_text(code_text, encoding="utf-8")

        ok_syntax, syntax_msg = _check_minizinc_syntax(model_path)
        if not ok_syntax:
            return {"ok": False, "error": f"MiniZinc syntax check failed: {syntax_msg}", "space": None, "num_solutions": 0}

        try:
            cmd = [
                "minizinc",
                "--solver", "gecode",
                "--all-solutions",
                "--num-solutions", str(solution_limit),
                "--time-limit", str(time_limit_sec),
                "--output-mode", "json",
                str(model_path),
            ]
            res = subprocess.run(cmd, capture_output=True, text=True)
        except FileNotFoundError:
            return {"ok": False, "error": "MiniZinc executable not found", "space": None, "num_solutions": 0}

        if res.returncode != 0 and not (res.stdout or "").strip():
            return {"ok": False, "error": (res.stderr or "").strip() or "MiniZinc run failed", "space": None, "num_solutions": 0}

        space = _extract_solutions_from_mzn_json_stream(res.stdout or "", decision_var_names)

        if len(space) == 0:
            status_txt = (res.stdout or "") + "\n" + (res.stderr or "")
            if "=====UNSATISFIABLE=====" in status_txt:
                return {"ok": True, "error": "", "space": set(), "num_solutions": 0}
            return {"ok": False, "error": "Could not parse MiniZinc solutions", "space": None, "num_solutions": 0}

        if len(space) >= solution_limit:
            return {
                "ok": True,
                "error": f"Reached solution_limit={solution_limit}; full space may be truncated",
                "space": space,
                "num_solutions": len(space),
            }

        return {"ok": True, "error": "", "space": space, "num_solutions": len(space)}

In [16]:
instance_dict = instance_to_dict(example_instance, data_instances)

ref_exec = execute_cpmpy_and_enumerate(
    code_text=reference_cp_model,
    instance_dict=instance_dict,
    decision_var_names=decision_variables,
    solution_limit=solution_limit,
    time_limit=time_limit_cpmpy_sec,
)
if not ref_exec["ok"]:
    raise RuntimeError(f"Reference model failed: {ref_exec['error']}")

reference_space = ref_exec["space"]
print(f"Reference complete solution space size: {len(reference_space)}")


Reference complete solution space size: 281


In [ ]:
max_feedback_rounds = 4

def classify_error_message(msg: str) -> tuple[str, str]:
    txt = str(msg or "")
    low = txt.lower()
    if "syntaxerror" in low or "indentationerror" in low or "syntax check failed" in low or "parse error" in low:
        category = "syntax_error"
    elif "timeout" in low or "time limit" in low:
        category = "timeout"
    elif "not found" in low:
        category = "environment"
    else:
        category = "runtime"

    compact = " | ".join([ln.strip() for ln in txt.splitlines() if ln.strip()][:3])
    return category, compact[:700]


def evaluate_candidate_code(language: str, code: str, claimed_label: str):
    lang_key = language.lower()
    if lang_key == "cpmpy":
        cand_exec = execute_cpmpy_and_enumerate(
            code_text=code,
            instance_dict=instance_dict,
            decision_var_names=decision_variables,
            solution_limit=solution_limit,
            time_limit=time_limit_cpmpy_sec,
        )
    elif lang_key == "minizinc":
        cand_exec = execute_minizinc_and_enumerate(
            code_text=code,
            decision_var_names=decision_variables,
            solution_limit=solution_limit,
            time_limit_sec=time_limit_minizinc_sec,
        )
    else:
        return {
            "exec_status": "not_evaluated",
            "candidate_space_size": None,
            "fp": None,
            "fn": None,
            "observed_label": "unknown",
            "label_match": None,
            "note": "Executor not implemented for this language",
            "error_category": "unsupported",
            "error_summary": "Executor not implemented for this language",
        }

    if not cand_exec["ok"]:
        err_category, err_summary = classify_error_message(cand_exec.get("error", ""))
        return {
            "exec_status": "non_executable",
            "candidate_space_size": None,
            "fp": None,
            "fn": None,
            "observed_label": "non-executable",
            "label_match": claimed_label == "non-executable",
            "note": cand_exec.get("error", ""),
            "error_category": err_category,
            "error_summary": err_summary,
        }

    cand_space = cand_exec["space"]
    observed, fp, fn = observed_label_from_sets(reference_space, cand_space)
    return {
        "exec_status": "ok",
        "candidate_space_size": len(cand_space),
        "fp": fp,
        "fn": fn,
        "observed_label": observed,
        "label_match": label_match_rule(claimed_label, observed, reference_space, cand_space),
        "note": "",
        "error_category": "",
        "error_summary": "",
    }


rows = []
candidate_models_loaded: dict[str, dict[str, dict[str, dict[str, Any]]]] = {}

for spec in model_specs:
    llm_name = spec.raw
    candidate_models_loaded[llm_name] = {}

    for language in language_labels:
        candidate_models_loaded[llm_name][language] = {}
        for claimed_label in correctness_labels:
            base = {"llm": llm_name, "language": language, "claimed_label": claimed_label}
            last_feedback = ""
            best_result = None
            attempt_history = []

            for attempt in range(1, max_feedback_rounds + 1):
                print(
                    f"Create+validate: llm={llm_name} | language={language} | label={claimed_label} | attempt={attempt}/{max_feedback_rounds}"
                )
                
                print(f"  feedback-> {last_feedback}")

                try:
                    code = generate_candidate_model(
                        spec=spec,
                        language=language,
                        label=claimed_label,
                        feedback_notes=last_feedback,
                    )
                except Exception as gen_e:
                    gen_msg = f"generation_error: {type(gen_e).__name__}: {gen_e}"
                    err_category, err_summary = classify_error_message(gen_msg)
                    print(f"  generation failed | category={err_category} | summary={err_summary}")
                    eval_result = {
                        "exec_status": "generation_failed",
                        "candidate_space_size": None,
                        "fp": None,
                        "fn": None,
                        "observed_label": "non-executable",
                        "label_match": claimed_label == "non-executable",
                        "note": gen_msg,
                        "error_category": err_category,
                        "error_summary": err_summary,
                    }
                    code = ""
                else:
                    eval_result = evaluate_candidate_code(language=language, code=code, claimed_label=claimed_label)
                    print(
                        f"  validation-> status={eval_result.get('exec_status')} | "
                        f"observed={eval_result.get('observed_label')} | match={eval_result.get('label_match')} | "
                        f"fp={eval_result.get('fp')} fn={eval_result.get('fn')}"
                    )
                    if eval_result.get("error_summary"):
                        print(f"  error-> {eval_result.get('error_summary')}")

                eval_result["attempts_used"] = attempt
                best_result = {"code": code, **eval_result}
                attempt_history.append({
                    "attempt": attempt,
                    "exec_status": eval_result.get("exec_status"),
                    "observed_label": eval_result.get("observed_label"),
                    "label_match": eval_result.get("label_match"),
                    "error_category": eval_result.get("error_category"),
                    "error_summary": eval_result.get("error_summary"),
                })

                if eval_result.get("label_match") is True:
                    print("  done-> target label matched")
                    break

                if eval_result.get("error_category") == "syntax_error":
                    last_feedback = (
                        f"Previous attempt had a syntax error ({eval_result.get('error_summary')}). "
                        "Fix syntax first and keep targeting the requested label behavior."
                    )
                    print("  retry-> syntax fix requested")
                else:
                    last_feedback = (
                        f"Previous candidate did not match requested label '{claimed_label}'. "
                        f"Observed='{eval_result.get('observed_label')}', exec_status='{eval_result.get('exec_status')}', "
                        f"fp={eval_result.get('fp')}, fn={eval_result.get('fn')}, "
                        f"error_category='{eval_result.get('error_category')}', "
                        f"error_summary='{eval_result.get('error_summary')}'. "
                        "Regenerate by adjusting constraints to better target the requested solution-space relation."
                    )
                    print("  retry-> solution-space mismatch, regenerating")

            candidate_models_loaded[llm_name][language][claimed_label] = {
                "ok": bool(best_result.get("label_match") is True),
                "code": best_result.get("code", ""),
                "attempts_used": best_result.get("attempts_used"),
                "final_observed_label": best_result.get("observed_label"),
                "final_label_match": best_result.get("label_match"),
                "final_exec_status": best_result.get("exec_status"),
                "final_error_category": best_result.get("error_category"),
                "final_error_summary": best_result.get("error_summary"),
                "feedback_last": last_feedback,
                "attempt_history": attempt_history,
            }

            rows.append({
                **base,
                "attempts_used": best_result.get("attempts_used"),
                "exec_status": best_result.get("exec_status"),
                "candidate_space_size": best_result.get("candidate_space_size"),
                "fp": best_result.get("fp"),
                "fn": best_result.get("fn"),
                "observed_label": best_result.get("observed_label"),
                "label_match": best_result.get("label_match"),
                "error_category": best_result.get("error_category"),
                "error_summary": best_result.get("error_summary"),
                "note": best_result.get("note", ""),
            })

    break

results_df = pd.DataFrame(rows).sort_values(["llm", "language", "claimed_label"]).reset_index(drop=True)
summary_df = (
    results_df
    .groupby(["language", "claimed_label", "observed_label"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["language", "claimed_label", "observed_label"])
)

display(results_df)
display(summary_df)

updated_candidate_json = output_dir / "candidate-models-feedback-loop.json"
updated_candidate_json.write_text(json.dumps(candidate_models_loaded, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved feedback-loop candidates: {updated_candidate_json}")

out_json_plain = output_dir / "candidate-models.json"
plain_payload = {
    llm: {
        lang: {
            lbl: {"ok": bool(meta.get("final_label_match") is True), "code": meta.get("code", "")}
            for lbl, meta in by_lbl.items()
        }
        for lang, by_lbl in by_lang.items()
    }
    for llm, by_lang in candidate_models_loaded.items()
}
out_json_plain.write_text(json.dumps(plain_payload, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved plain candidates: {out_json_plain}")

out_csv = output_dir / "candidate-label-verification-universal-with-minizinc.csv"
results_df.to_csv(out_csv, index=False)
print(f"Saved report: {out_csv}")

Create+validate: llm=openai/gpt-5.1-codex-mini | language=minizinc | label=equivalent | attempt=1/4
  feedback-> 
  validation-> status=non_executable | observed=non-executable | match=False | fp=None fn=None
  error-> MiniZinc syntax check failed: C:\Users\nmofa\AppData\Local\Temp\tmp12gqrtlo\candidate.mzn:9.46: | (if j > 1 then bool2int(h[i, j-1]) else 0) | ^
  retry-> syntax fix requested
Create+validate: llm=openai/gpt-5.1-codex-mini | language=minizinc | label=equivalent | attempt=2/4
  feedback-> Previous attempt had a syntax error (MiniZinc syntax check failed: C:\Users\nmofa\AppData\Local\Temp\tmp12gqrtlo\candidate.mzn:9.46: | (if j > 1 then bool2int(h[i, j-1]) else 0) | ^). Fix syntax first and keep targeting the requested label behavior.
  validation-> status=non_executable | observed=non-executable | match=False | fp=None fn=None
  error-> MiniZinc syntax check failed: C:\Users\nmofa\AppData\Local\Temp\tmp5vhldg7x\candidate.mzn:8.46: | (if j > 1 then bool2int(h[i, j-1]) else

,llm,language,claimed_label,attempts_used,exec_status,candidate_space_size,fp,fn,observed_label,label_match,error_category,error_summary,note
0,openai/gpt-5.1-codex-mini,CPMpy,equivalent,1,ok,281.0,0.0,0.0,equivalent,True,,,
1,openai/gpt-5.1-codex-mini,CPMpy,incomplete,1,ok,132.0,0.0,149.0,incomplete,True,,,
2,openai/gpt-5.1-codex-mini,CPMpy,non-executable,1,non_executable,NaN,NaN,NaN,non-executable,True,runtime,TypeError: boolvar() got an unexpected keyword...,TypeError: boolvar() got an unexpected keyword...
3,openai/gpt-5.1-codex-mini,CPMpy,status-only correct,1,ok,100000.0,99998.0,279.0,unsound-incomplete,True,,,
4,openai/gpt-5.1-codex-mini,CPMpy,unsound,4,ok,100000.0,99998.0,279.0,unsound-incomplete,False,,,
5,openai/gpt-5.1-codex-mini,CPMpy,unsound-incomplete,2,ok,100000.0,99998.0,279.0,unsound-incomplete,True,,,
6,openai/gpt-5.1-codex-mini,minizinc,equivalent,4,ok,206.0,0.0,75.0,incomplete,False,,,
7,openai/gpt-5.1-codex-mini,minizinc,incomplete,2,ok,132.0,0.0,149.0,incomplete,True,,,
8,openai/gpt-5.1-codex-mini,minizinc,non-executable,1,non_executable,NaN,NaN,NaN,non-executable,True,syntax_error,MiniZinc syntax check failed: C:\Users\nmofa\A...,MiniZinc syntax check failed: C:\Users\nmofa\A...
9,openai/gpt-5.1-codex-mini,minizinc,status-only correct,1,ok,206.0,206.0,281.0,unsound-incomplete,True,,,


,language,claimed_label,observed_label,count
0,CPMpy,equivalent,equivalent,1
1,CPMpy,incomplete,incomplete,1
2,CPMpy,non-executable,non-executable,1
3,CPMpy,status-only correct,unsound-incomplete,1
4,CPMpy,unsound,unsound-incomplete,1
5,CPMpy,unsound-incomplete,unsound-incomplete,1
6,minizinc,equivalent,incomplete,1
7,minizinc,incomplete,incomplete,1
8,minizinc,non-executable,non-executable,1
9,minizinc,status-only correct,unsound-incomplete,1


Saved feedback-loop candidates: data-storage\candidates-domino_tiling\candidate-models-feedback-loop.json
Saved plain candidates: data-storage\candidates-domino_tiling\candidate-models.json
Saved report: data-storage\candidates-domino_tiling\candidate-label-verification-universal-with-minizinc.csv


In [18]:
print(reference_cp_model)


from cpmpy import *

h = boolvar(shape=(m, n - 1), name="h")  # horizontal domino starts
v = boolvar(shape=(m - 1, n), name="v")  # vertical domino starts

model = Model()
for i in range(m):
    for j in range(n):
        cover = []
        if j > 0:
            cover.append(h[i, j - 1])
        if j < n - 1:
            cover.append(h[i, j])
        if i > 0:
            cover.append(v[i - 1, j])
        if i < m - 1:
            cover.append(v[i, j])
        model += (sum(cover) == 1)

if model.solve():
    print(h.value())
    print(v.value())



In [19]:
for lang in language_labels:
    print(f"\n\n\n****************************************** {lang}  **********************************************************")
    for label in correctness_labels:
        print(f"\n\n\n--------------------------  {label}  ---------------------------")
        print(plain_payload['openai/gpt-5.1-codex-mini'][lang][label]['code'])




****************************************** minizinc  **********************************************************



--------------------------  equivalent  ---------------------------
int: m = 4;
int: n = 6;

array[1..m, 1..n-1] of var bool: h;
array[1..m-1, 1..n] of var bool: v;

constraint
  forall(i in 1..m, j in 1..n)(
    bool2int(if j > 1 then h[i, j - 1] else false endif) +
    bool2int(if j < n then h[i, j] else false endif) +
    bool2int(if i > 1 then v[i - 1, j] else false endif) +
    bool2int(if i < m then v[i, j] else false endif) == 1
  );

solve satisfy;

output [
  "h = ", show(h), "\n",
  "v = ", show(v), "\n"
];



--------------------------  unsound  ---------------------------
int: m = 4;
int: n = 6;

array[1..m, 1..n - 1] of var bool: h;
array[1..m - 1, 1..n] of var bool: v;

constraint
  forall(i in 1..m, j in 1..n) (
    bool2int(if j > 1 then h[i, j - 1] else false endif) +
    bool2int(if j < n then h[i, j] else false endif) +
    bool2int(if i > 1 then v[i 

In [20]:
from cpmpy import *

m = 4
n = 6

h = boolvar(shape=(m, n - 1), name="h")
v = boolvar(shape=(m - 1, n), name="v")
model = Model()

if (m * n) % 2 != 0:
    model += (0 == 1)

for i in range(m):
    for j in range(n - 1):
        model += (h[i, j] == (j % 2 == 0))

for i in range(m - 1):
    for j in range(n):
        if (n % 2 == 1) and (j == n - 1) and (i % 2 == 0):
            model += (v[i, j] == 1)
        else:
            model += (v[i, j] == 0)

model.solveAll(display=[h,v])

[[[True, False, True, False, True], [True, False, True, False, True], [True, False, True, False, True], [True, False, True, False, True]], [[False, False, False, False, False, False], [False, False, False, False, False, False], [False, False, False, False, False, False]]]


1

In [21]:
minizinz_code = ''' int: m = 4;
int: n = 6;

array[1..m, 1..n-1] of var bool: h;
array[1..m-1, 1..n] of var bool: v;

constraint
  forall(i in 1..m, j in 1..n) (
    let {
      var int: cover_h_right = if j < n then bool2int(h[i,j]) else 0 endif,
      var int: cover_h_left  = if j > 1 then bool2int(h[i,j-1]) else 0 endif,
      var int: cover_v_down = if i < m then bool2int(v[i,j]) else 0 endif,
      var int: cover_v_up   = if i > 1 then bool2int(v[i-1,j]) else 0 endif
    } in
    cover_h_right + cover_h_left + cover_v_down + cover_v_up == 1
  );

solve satisfy;


'''

import subprocess
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp_dir:
    model_path = Path(tmp_dir) / "domino_tiling_all_solutions.mzn"
    model_path.write_text(minizinz_code, encoding="utf-8")

    # Enumerate the full solution space
    result = subprocess.run(
        ["minizinc", "--solver", "gecode", "--all-solutions", str(model_path)],
        capture_output=True,
        text=True,
    )

if result.returncode != 0:
    raise RuntimeError(f"MiniZinc failed:\n{result.stderr}")

raw_output = result.stdout.strip()
solution_blocks = [
    block.strip()
    for block in raw_output.split("----------")
    if block.strip() and "==========" not in block
]

print(f"Total solutions: {len(solution_blocks)}")
for idx, block in enumerate(solution_blocks, start=1):
    print(f"\n--- Solution {idx} ---")
    print(block)


Total solutions: 281

--- Solution 1 ---
h = 
[| true, false, true, false, true
 | true, false, true, false, true
 | true, false, true, false, true
 | true, false, true, false, true
 |];
v = 
[| false, false, false, false, false, false
 | false, false, false, false, false, false
 | false, false, false, false, false, false
 |];

--- Solution 2 ---
h = 
[| false, false, true, false, true
 | false, false, true, false, true
 |  true, false, true, false, true
 |  true, false, true, false, true
 |];
v = 
[|  true,  true, false, false, false, false
 | false, false, false, false, false, false
 | false, false, false, false, false, false
 |];

--- Solution 3 ---
h = 
[| false,  true, false, false, true
 | false,  true, false, false, true
 |  true, false,  true, false, true
 |  true, false,  true, false, true
 |];
v = 
[|  true, false, false,  true, false, false
 | false, false, false, false, false, false
 | false, false, false, false, false, false
 |];

--- Solution 4 ---
h = 
[| true, false, fa

## LLM-as-Judge

In [ ]:
model_specs

[ModelSpec(provider='openai', name='gpt-5.1-codex-mini', tag=None, raw='openai/gpt-5.1-codex-mini'),
 ModelSpec(provider='openai', name='gpt-4o-mini', tag=None, raw='openai/gpt-4o-mini')]

In [ ]:
plain_payload[generator_spec.raw].items()

dict_items([('minizinc', {'equivalent': {'ok': False, 'code': 'int: m = 4;\nint: n = 6;\n\narray[1..m, 1..n-1] of var bool: h;\narray[1..m-1, 1..n] of var bool: v;\n\nconstraint\n  forall(i in 1..m, j in 1..n)(\n    bool2int(if j > 1 then h[i, j - 1] else false endif) +\n    bool2int(if j < n then h[i, j] else false endif) +\n    bool2int(if i > 1 then v[i - 1, j] else false endif) +\n    bool2int(if i < m then v[i, j] else false endif) == 1\n  );\n\nsolve satisfy;\n\noutput [\n  "h = ", show(h), "\\n",\n  "v = ", show(v), "\\n"\n];'}, 'unsound': {'ok': False, 'code': 'int: m = 4;\nint: n = 6;\n\narray[1..m, 1..n - 1] of var bool: h;\narray[1..m - 1, 1..n] of var bool: v;\n\nconstraint\n  forall(i in 1..m, j in 1..n) (\n    bool2int(if j > 1 then h[i, j - 1] else false endif) +\n    bool2int(if j < n then h[i, j] else false endif) +\n    bool2int(if i > 1 then v[i - 1, j] else false endif) +\n    bool2int(if i < m then v[i, j] else false endif) >= 1\n  );\n\nsolve satisfy;\n\noutput [\

In [38]:
# Judge models: small -> stronger/recent
judge_models_set = [
    "openai/gpt-4o-mini",
    "openai/gpt-5.4-mini",
]
judge_specs = [ModelSpec.from_string(m) for m in judge_models_set]


def normalize_label(text: str) -> str:
    lowered = (text or "").lower()
    for lbl in sorted(correctness_labels, key=len, reverse=True):
        if lbl.lower() in lowered:
            return lbl
    return "unknown"


def parse_judge_json(raw_text: str) -> dict[str, Any]:
    txt = (raw_text or "").strip()
    try:
        obj = json.loads(txt)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    match = re.search(r"\{.*\}", txt, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    return {
        "label": normalize_label(txt),
        "confidence": None,
        "rationale": txt[:300],
    }


def direct_llm_judge(spec: ModelSpec, candidate_code: str, language: str) -> tuple[str, dict[str, Any], str]:
    llm = get_openrouter_llm(spec)
    labels_text = ", ".join(correctness_labels)
    prompt = f"""
You are an expert CP model judge.
Classify the candidate with exactly one allowed correctness label.

Allowed labels: {labels_text}

Reasoning protocol (perform internally before answering):
1) Check executability for the given language (syntax/API shape + obvious runtime issues).
2) Infer intended semantics from the problem description.
3) Compare candidate constraints to intended semantics.
4) Decide whether candidate is equivalent, unsound, incomplete, unsound-incomplete, status-only correct, or non-executable.

Output format (STRICT JSON only):
{{"label": "<one allowed label>", "confidence": <0..1>, "rationale": "<1-3 short sentences>"}}

Problem description:
{problem_description}

Candidate ({language}):
{candidate_code}
""".strip()

    raw = llm_response_to_text(llm.invoke(prompt)).strip()
    parsed = parse_judge_json(raw)
    pred_label = normalize_label(str(parsed.get("label", "")))
    return pred_label, parsed, raw


# Fallback: load plain payload from disk if not in memory
if "plain_payload" not in globals():
    in_json = output_dir / "candidate-models.json"
    plain_payload = json.loads(in_json.read_text(encoding="utf-8"))


generator_key = next(iter(plain_payload.keys()))
judge_rows = []

for judge_spec in judge_specs:
    print(f"\nRunning judge: {judge_spec.raw}")
    try:
        for language, by_label in plain_payload[generator_key].items():
            for claimed_label, meta in by_label.items():
                candidate_code = meta.get("code", "")
                status_label = meta.get("ok", "" )
                if not candidate_code.strip() or status_label==False:
                    print(f"Ecluded! {language} , {claimed_label}")
                    continue

                pred_label, parsed, raw_reply = direct_llm_judge(judge_spec, candidate_code, language)
                judge_rows.append(
                    {
                        "judge_llm": judge_spec.raw,
                        "generator_llm": generator_key,
                        "language": language,
                        "claimed_label": claimed_label,
                        "predicted_label": pred_label,
                        "match": pred_label == claimed_label,
                        "confidence": parsed.get("confidence"),
                        "rationale": parsed.get("rationale", ""),
                        "raw_reply": raw_reply,
                    }
                )
    except Exception as e:
        print(f"Judge failed for {judge_spec.raw}: {type(e).__name__}: {e}")

judge_df = pd.DataFrame(judge_rows)
display(judge_df)

if not judge_df.empty:
    model_summary_df = (
        judge_df.groupby("judge_llm", dropna=False)["match"]
        .mean()
        .reset_index(name="accuracy")
        .sort_values("accuracy", ascending=False)
    )
    display(model_summary_df)
else:
    print("No judge outputs produced.")


Running judge: openai/gpt-4o-mini
Ecluded! minizinc , equivalent
Ecluded! minizinc , unsound
Ecluded! CPMpy , unsound

Running judge: openai/gpt-5.4-mini
Ecluded! minizinc , equivalent
Ecluded! minizinc , unsound
Ecluded! CPMpy , unsound


,judge_llm,generator_llm,language,claimed_label,predicted_label,match,confidence,rationale,raw_reply
0,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,minizinc,incomplete,unsound,False,0.80,The candidate does not ensure all cells are co...,"{""label"": ""unsound"", ""confidence"": 0.8, ""ratio..."
1,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,minizinc,unsound-incomplete,unsound-incomplete,True,0.80,The candidate attempts to model the tiling of ...,"{""label"": ""unsound-incomplete"", ""confidence"": ..."
2,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,minizinc,non-executable,unsound,False,0.80,The candidate attempts to ensure that each squ...,"{""label"": ""unsound"", ""confidence"": 0.8, ""ratio..."
3,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,minizinc,status-only correct,unsound-incomplete,False,0.80,The candidate correctly identifies that tiling...,"{""label"": ""unsound-incomplete"", ""confidence"": ..."
4,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,CPMpy,equivalent,unsound-incomplete,False,0.80,The candidate does not ensure a complete and v...,"{""label"": ""unsound-incomplete"", ""confidence"": ..."
5,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,CPMpy,incomplete,unsound-incomplete,False,0.80,The candidate's logic for covering the board i...,"```json\n{""label"": ""unsound-incomplete"", ""conf..."
6,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,CPMpy,unsound-incomplete,status-only correct,False,0.80,The candidate code implements a constraint pro...,"{""label"": ""status-only correct"", ""confidence"":..."
7,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,CPMpy,non-executable,status-only correct,False,0.80,The candidate attempts to model the problem us...,"{""label"": ""status-only correct"", ""confidence"":..."
8,openai/gpt-4o-mini,openai/gpt-5.1-codex-mini,CPMpy,status-only correct,unsound,False,0.80,The candidate fails to ensure a valid tiling o...,"{""label"": ""unsound"", ""confidence"": 0.8, ""ratio..."
9,openai/gpt-5.4-mini,openai/gpt-5.1-codex-mini,minizinc,incomplete,equivalent,False,0.93,The MiniZinc model is executable and enforces ...,"{""label"":""equivalent"",""confidence"":0.93,""ratio..."


,judge_llm,accuracy
1,openai/gpt-5.4-mini,0.222222
0,openai/gpt-4o-mini,0.111111


In [1]:
print("int: m = 4;\nint: n = 6;\n\narray[1..m, 1..n-1] of var bool: h;\narray[1..m-1, 1..n] of var bool: v;\n\nconstraint\n  forall(i in 1..m, j in 1..n) (\n    let {\n      array[int] of var int: covers = \n        (if j > 1 then [bool2int(h[i, j-1])] else [])\n      ++ (if j < n then [bool2int(h[i, j])] else [])\n      ++ (if i > 1 then [bool2int(v[i-1, j])] else [])\n      ++ (if i < m then [bool2int(v[i, j])] else [])\n    } in\n      sum(covers) == 1\n  );\n\nsolve satisfy;\n\noutput [\n  \"h =\\n\",\n  show(h), \"\\n\",\n  \"v =\\n\",\n  show(v), \"\\n\"\n];")

int: m = 4;
int: n = 6;

array[1..m, 1..n-1] of var bool: h;
array[1..m-1, 1..n] of var bool: v;

constraint
  forall(i in 1..m, j in 1..n) (
    let {
      array[int] of var int: covers = 
        (if j > 1 then [bool2int(h[i, j-1])] else [])
      ++ (if j < n then [bool2int(h[i, j])] else [])
      ++ (if i > 1 then [bool2int(v[i-1, j])] else [])
      ++ (if i < m then [bool2int(v[i, j])] else [])
    } in
      sum(covers) == 1
  );

solve satisfy;

output [
  "h =\n",
  show(h), "\n",
  "v =\n",
  show(v), "\n"
];
